### 1. Identify Target Banks and Retrieve FDIC Certificate Numbers (CERTs)
To query financial data for specific banks using the FDIC API, we first need to obtain their standard unique identifiers (called Certificate Numbers or `CERT`). This script searches for the certificate numbers of your 7 target banks.

In [1]:
import requests
import pandas as pd

target_banks = [
    "First Horizon Bank",
    "Huntington National Bank",
    "KeyBank National Association",
    "Manufacturers and Traders Trust Company",
    "Old National Bank",
    "SouthState Bank",
    "Glacier Bank"
]

def get_bank_certs(bank_names):
    url = "https://banks.data.fdic.gov/api/institutions"
    certs = {}
    for bank in bank_names:
        # Hardcode Huntington National Bank's CERT to 6560
        if bank == "Huntington National Bank":
            certs[bank] = 6560
            print(f"Found: {bank} -> CERT: 6560")
            continue

        # Try exact search
        params = {
            'filters': f'NAME:"{bank}" AND ACTIVE:1',
            'fields': 'CERT,NAME,CITY,STNAME',
            'limit': 5
        }
        response = requests.get(url, params=params).json()
        if response.get('data'):
            match = response['data'][0]['data']
            certs[bank] = match['CERT']
            print(f"Found: {bank} -> CERT: {match['CERT']} ({match['CITY']}, {match['STNAME']})")
        else:
            # Looser check: try alternative space spelling (e.g. "South State Bank" for "SouthState Bank")
            search_term = bank.replace("SouthState", "South State")
            params['filters'] = f'NAME:"{search_term}" AND ACTIVE:1'
            response = requests.get(url, params=params).json()
            if response.get('data'):
                match = response['data'][0]['data']
                certs[bank] = match['CERT']
                print(f"Found (Alternative Search): {bank} -> CERT: {match['CERT']} ({match['CITY']}, {match['STNAME']})")
            else:
                # Looser match overall
                params['filters'] = f'NAME:*{bank.split()[0]}* AND ACTIVE:1'
                response = requests.get(url, params=params).json()
                if response.get('data'):
                    match = response['data'][0]['data']
                    certs[bank] = match['CERT']
                    print(f"Found (Fuzzy Search): {bank} -> CERT: {match['CERT']} ({match['NAME']})")
                else:
                    print(f"Could not find CERT for: {bank}")
    return certs

bank_certs_map = get_bank_certs(target_banks)
print("\nMapping completed:", bank_certs_map)

Found: First Horizon Bank -> CERT: 4977 (Memphis, Tennessee)
Found: Huntington National Bank -> CERT: 6560
Found: KeyBank National Association -> CERT: 17534 (Cleveland, Ohio)
Found: Manufacturers and Traders Trust Company -> CERT: 588 (Buffalo, New York)
Found: Old National Bank -> CERT: 3832 (Evansville, Indiana)
Found (Fuzzy Search): SouthState Bank -> CERT: 33555 (SouthState Bank, National Association)
Found: Glacier Bank -> CERT: 30788 (Kalispell, Montana)

Mapping completed: {'First Horizon Bank': 4977, 'Huntington National Bank': 6560, 'KeyBank National Association': 17534, 'Manufacturers and Traders Trust Company': 588, 'Old National Bank': 3832, 'SouthState Bank': 33555, 'Glacier Bank': 30788}


### 2. Quarterly FDIC Financials Dataset Retrieval and Filtering Workflow

The cell below defines the core automated workflow. It generates a timeline of quarters from `2008-Q1` through `2026-Q4`, calls the FDIC Financials endpoint (`https://banks.data.fdic.gov/api/financials`), retrieves the matching reporting fields, filters them, and structures the consolidated output into a final Pandas DataFrame.

In [2]:
import requests
import pandas as pd

def diagnostic_and_fetch(bank_map, start_date="2008-01-01", end_date="2026-12-31"):
    url = "https://banks.data.fdic.gov/api/financials"

    # Convert standard CERT list into explicit OR conditions
    cert_list = list(bank_map.values())
    cert_filter_str = " OR ".join([f"CERT:{cert}" for cert in cert_list])
    cert_to_name = {v: k for k, v in bank_map.items()}

    # Since REPDTE in raw data looks like '19840331', format dates as YYYYMMDD strings
    start_date_str = start_date.replace("-", "")
    end_date_str = end_date.replace("-", "")

    params = {
        'filters': f"({cert_filter_str}) AND REPDTE:[{start_date_str} TO {end_date_str}]",
        'fields': "CERT,REPDTE,EINTEXP,DEP",
        'limit': 2000
    }

    print(f"Attempting range query using REPDTE for CERTs between {start_date_str} and {end_date_str}...")
    try:
        res = requests.get(url, params=params)
        if res.status_code == 200:
            data = res.json().get('data', [])
            all_records = []
            for item in data:
                row = item.get('data', {})
                cert = row.get('CERT')
                repdte = row.get('REPDTE')
                intexp = row.get('EINTEXP', 0)
                dep = row.get('DEP', 0)

                if cert:
                    cert_val = int(cert)
                    all_records.append({
                        'bank_name': cert_to_name.get(cert_val, f"Unknown CERT {cert}"),
                        'cert_number': cert_val,
                        'quarter': repdte,
                        'interest_expense': intexp,
                        'total_domestic_deposits': dep
                    })

            df = pd.DataFrame(all_records)
            if not df.empty:
                df = df.sort_values(by=['bank_name', 'quarter']).reset_index(drop=True)
            return df
        else:
            print(f"Range query failed with status: {res.status_code}. API Response: {res.text}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching financials: {e}")
        return pd.DataFrame()

# Ensure bank_certs_map is defined in this session
bank_certs_map = {
    'First Horizon Bank': 4977,
    'Huntington National Bank': 6560,
    'KeyBank National Association': 17534,
    'Manufacturers and Traders Trust Company': 588,
    'Old National Bank': 3832,
    'SouthState Bank': 33555,
    'Glacier Bank': 30788
}

# Execute diagnostics and retrieval
consolidated_df = diagnostic_and_fetch(bank_certs_map)

# Preview output
if not consolidated_df.empty:
    display(consolidated_df.head(10))
else:
    print("No results retrieved. Please check the date formats or filter string configuration.")

Attempting range query using REPDTE for CERTs between 20080101 and 20261231...


,bank_name,cert_number,quarter,interest_expense,total_domestic_deposits
0,First Horizon Bank,4977,20080331,243847,16341861
1,First Horizon Bank,4977,20080630,417040,15261402
2,First Horizon Bank,4977,20080930,573720,13906343
3,First Horizon Bank,4977,20081231,696532,14492073
4,First Horizon Bank,4977,20090331,70423,14917342
5,First Horizon Bank,4977,20090630,123602,15183704
6,First Horizon Bank,4977,20090930,166186,14421976
7,First Horizon Bank,4977,20091231,204596,15035386
8,First Horizon Bank,4977,20100331,36473,15254978
9,First Horizon Bank,4977,20100630,72128,15210877


In [4]:
import pandas as pd

# Ensure we have a fresh copy of the retrieved FDIC dataset
raw_cumulative_df = consolidated_df.copy()

# Convert the 'quarter' column (which is YYYYMMDD string or integer format) into a conventional datetime type
raw_cumulative_df['quarter_dt'] = pd.to_datetime(raw_cumulative_df['quarter'].astype(str), format='%Y%m%d', errors='coerce')

# Verify conversion
display(raw_cumulative_df[['bank_name', 'quarter', 'quarter_dt']].head(10))

,bank_name,quarter,quarter_dt
0,First Horizon Bank,20080331,2008-03-31
1,First Horizon Bank,20080630,2008-06-30
2,First Horizon Bank,20080930,2008-09-30
3,First Horizon Bank,20081231,2008-12-31
4,First Horizon Bank,20090331,2009-03-31
5,First Horizon Bank,20090630,2009-06-30
6,First Horizon Bank,20090930,2009-09-30
7,First Horizon Bank,20091231,2009-12-31
8,First Horizon Bank,20100331,2010-03-31
9,First Horizon Bank,20100630,2010-06-30


In [5]:
import pandas as pd

# Use raw_cumulative_df with the converted datetime objects
df = raw_cumulative_df.copy()

# Extract year and month directly from the conventional datetime object
df['year'] = df['quarter_dt'].dt.year

df['month'] = df['quarter_dt'].dt.month

# 2. Methodically separate the dataframe into 4 distinct quarters based on the converted month
df_q1 = df[df['month'] == 3].copy()
df_q2 = df[df['month'] == 6].copy()
df_q3 = df[df['month'] == 9].copy()
df_q4 = df[df['month'] == 12].copy()

# Create reference dataframes containing the unaltered cumulative interest expenses for subtraction
df_q1_ref = df_q1[['bank_name', 'year', 'interest_expense']].rename(columns={'interest_expense': 'q1_interest_cumulative'})
df_q2_ref = df_q2[['bank_name', 'year', 'interest_expense']].rename(columns={'interest_expense': 'q2_interest_cumulative'})
df_q3_ref = df_q3[['bank_name', 'year', 'interest_expense']].rename(columns={'interest_expense': 'q3_interest_cumulative'})

# 3. Sequentially carry out the subtractions using vector-based merges
# Step A: Decumulate Q4 -> Subtract Q3 cumulative from Q4
df_q4 = df_q4.merge(df_q3_ref, on=['bank_name', 'year'], how='left')
df_q4['interest_expense'] = df_q4['interest_expense'] - df_q4['q3_interest_cumulative'].fillna(0)
df_q4 = df_q4.drop(columns=['q3_interest_cumulative'])

# Step B: Decumulate Q3 -> Subtract Q2 cumulative from Q3
df_q3 = df_q3.merge(df_q2_ref, on=['bank_name', 'year'], how='left')
df_q3['interest_expense'] = df_q3['interest_expense'] - df_q3['q2_interest_cumulative'].fillna(0)
df_q3 = df_q3.drop(columns=['q2_interest_cumulative'])

# Step C: Decumulate Q2 -> Subtract Q1 cumulative from Q2
df_q2 = df_q2.merge(df_q1_ref, on=['bank_name', 'year'], how='left')
df_q2['interest_expense'] = df_q2['interest_expense'] - df_q2['q1_interest_cumulative'].fillna(0)
df_q2 = df_q2.drop(columns=['q1_interest_cumulative'])

# 4. Recombine and clean up dataset
consolidated_df = pd.concat([df_q1, df_q2, df_q3, df_q4], ignore_index=True)

# Group by BOTH 'bank_name' and the calendar year to ensure the average is calculated per-bank
consolidated_df["annual_average_deposits"] = consolidated_df.groupby(['bank_name', consolidated_df['quarter_dt'].dt.year])["total_domestic_deposits"].transform("mean")

# Ensure no negative interest expenses are produced due to minor reporting adjustments
consolidated_df['interest_expense'] = consolidated_df['interest_expense'].clip(lower=0)

# Recalculate deposit rate on correct, decumulated period-specific values
consolidated_df['deposit_rate'] = (consolidated_df['interest_expense'] / consolidated_df['annual_average_deposits']) * 100 * 4

# Format date into standard YYYY-MM-DD string
consolidated_df['quarter'] = consolidated_df['quarter_dt'].dt.strftime('%Y-%m-%d')

# Sort chronologically to match the original dataset's size and order
consolidated_df = consolidated_df.sort_values(by=['bank_name', 'quarter']).reset_index(drop=True)

# Drop helper columns
consolidated_df = consolidated_df.drop(columns=['quarter_dt', 'year', 'month'])

# Explicitly reorder columns to ensure 'cert_number' is adjacent to 'bank_name'
cols = ['bank_name', 'cert_number'] + [col for col in consolidated_df.columns if col not in ['bank_name', 'cert_number']]
consolidated_df = consolidated_df[cols]

# Display the final sanitized results
display(consolidated_df.head(10))

,bank_name,cert_number,quarter,interest_expense,total_domestic_deposits,annual_average_deposits,deposit_rate
0,First Horizon Bank,4977,2008-03-31,243847,16341861,15000419.75,6.502405
1,First Horizon Bank,4977,2008-06-30,173193,15261402,15000419.75,4.618351
2,First Horizon Bank,4977,2008-09-30,156680,13906343,15000419.75,4.178016
3,First Horizon Bank,4977,2008-12-31,122812,14492073,15000419.75,3.274895
4,First Horizon Bank,4977,2009-03-31,70423,14917342,14889602.00,1.891871
5,First Horizon Bank,4977,2009-06-30,53179,15183704,14889602.00,1.428621
6,First Horizon Bank,4977,2009-09-30,42584,14421976,14889602.00,1.143993
7,First Horizon Bank,4977,2009-12-31,38410,15035386,14889602.00,1.031861
8,First Horizon Bank,4977,2010-03-31,36473,15254978,15282998.50,0.954603
9,First Horizon Bank,4977,2010-06-30,35655,15210877,15282998.50,0.933194


### 3. Save the Consolidated Dataset
Finally, we will save the extracted results as a consolidated CSV file locally within your workspace environment.

In [6]:
output_path = "/content/drive/MyDrive/Deposit Beta/consolidated_fdic_quarterly_financials_2008-2026.csv"
consolidated_df.to_csv(output_path, index=False)
print(f"Successfully consolidated and saved file to: {output_path}")

Successfully consolidated and saved file to: /content/drive/MyDrive/Deposit Beta/consolidated_fdic_quarterly_financials_2008-2026.csv
